# Step 1: Setup and Data Loading

This section installs the required packages, verifies the text splitter imports, loads API keys from `.env`, transcribes the podcast, and loads the PDF into memory.

In [ ]:
# Install the small set of packages needed for Step 1.
# This is separate so the notebook can fail fast if dependencies are missing.
%pip install -q openai python-dotenv langchain-text-splitters pypdf tiktoken

In [ ]:
# Import the text splitter classes that will be used later in the lab.
# The expected first visible output is a clean import with no errors.
from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter

print("Imported RecursiveCharacterTextSplitter and TokenTextSplitter successfully.")

In [ ]:
# Load environment variables from the local .env file.
# This keeps the API key out of the notebook and makes the setup reproducible.
from pathlib import Path
import os

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY was not loaded from .env"

client = OpenAI()
print("Environment loaded and OpenAI client initialized.")

In [ ]:
# Transcribe the podcast audio into a local text file.
# Saving the transcript locally makes the source material inspectable and reusable.
podcast_path = Path("data/CVX_manufacturing_podcast.mp3")
transcript_path = Path("data/CVX_manufacturing_podcast_transcript.txt")

assert podcast_path.exists(), f"Missing podcast file: {podcast_path}"

transcription_model = os.getenv("OPENAI_TRANSCRIPTION_MODEL", "whisper-1")
with podcast_path.open("rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model=transcription_model,
        file=audio_file,
    )

transcript_text = transcript.text
transcript_path.write_text(transcript_text, encoding="utf-8")

print(f"Transcript saved to {transcript_path}")
print(f"Transcript characters: {len(transcript_text)}")

In [ ]:
# Load the PDF into plain text so it can be chunked later.
# We keep the original file and also create a text string for inspection.
pdf_path = Path("data/The AI Ladder.pdf")

assert pdf_path.exists(), f"Missing PDF file: {pdf_path}"

pdf_reader = PdfReader(str(pdf_path))
pdf_text = "\n".join(page.extract_text() or "" for page in pdf_reader.pages)

print(f"PDF pages: {len(pdf_reader.pages)}")
print(f"PDF text characters: {len(pdf_text)}")

# Step 2: Fixed-Size Chunking

This section uses `CharacterTextSplitter` to compare fixed-size chunks across the podcast transcript and the PDF for multiple chunk sizes and overlaps.

In [ ]:
# Re-load the two source texts so this cell stays testable even if run on its own.
# The transcript should already exist from Step 1, but we read it again here to keep Step 2 isolated.
from pathlib import Path

from langchain_text_splitters import CharacterTextSplitter
from pypdf import PdfReader

transcript_path = Path("data/CVX_manufacturing_podcast_transcript.txt")
pdf_path = Path("data/The AI Ladder.pdf")

assert transcript_path.exists(), f"Missing transcript file: {transcript_path}"
assert pdf_path.exists(), f"Missing PDF file: {pdf_path}"

podcast_text = transcript_path.read_text(encoding="utf-8")
pdf_text = "\n".join(page.extract_text() or "" for page in PdfReader(str(pdf_path)).pages)

sizes = [500, 1000, 2000]
overlaps = [0, 50, 100]

def make_chunks(text, chunk_size, chunk_overlap):
    # This splitter creates fixed-width character windows.
    # It is intentionally simple so we can observe how it behaves on two very different document types.
    splitter = CharacterTextSplitter(
        separator="\n\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )
    return splitter.split_text(text)

def summarize_chunks(label, text):
    print(f"\n=== {label} ===")
    for chunk_size in sizes:
        for chunk_overlap in overlaps:
            chunks = make_chunks(text, chunk_size, chunk_overlap)
            lengths = [len(c) for c in chunks]
            print(
                f"size={chunk_size:4d} overlap={chunk_overlap:3d} | "
                f"chunks={len(chunks):3d} | "
                f"avg_len={sum(lengths) / len(lengths):7.1f} | "
                f"first={repr(chunks[0][:80]) if chunks else '[]'}"
            )

            # Print a short tail sample for one representative configuration.
            # This makes it easy to inspect whether a sentence or paragraph boundary was cut.
            if chunk_size == 500 and chunk_overlap == 0 and chunks:
                print(f"  sample_end={repr(chunks[0][-80:])}")

summarize_chunks("Podcast transcript", podcast_text)
summarize_chunks("PDF document", pdf_text)

# Step 3: Recursive Character Chunking

This section uses `RecursiveCharacterTextSplitter` with token-aware chunk lengths so the splitter can fall back through a separator priority list and try to keep sentences and paragraphs together.

In [ ]:
# Compare recursive chunking against the fixed-size baseline from Step 2.
# The chunk_size values below are token counts, so we use a tokenizer-backed length function.
from pathlib import Path

import tiktoken
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from pypdf import PdfReader

transcript_path = Path("data/CVX_manufacturing_podcast_transcript.txt")
pdf_path = Path("data/The AI Ladder.pdf")

assert transcript_path.exists(), f"Missing transcript file: {transcript_path}"
assert pdf_path.exists(), f"Missing PDF file: {pdf_path}"

podcast_text = transcript_path.read_text(encoding="utf-8")
pdf_text = "\n".join(page.extract_text() or "" for page in PdfReader(str(pdf_path)).pages)

# Use the OpenAI tokenizer encoding so chunk sizes are measured in tokens rather than raw characters.
encoding = tiktoken.get_encoding("cl100k_base")
token_len = lambda text: len(encoding.encode(text))

chunk_sizes = [500, 1000, 2000]
chunk_overlap = 100
separator_sets = {
    "paragraph_first": ["\n\n", "\n", ". ", " ", ""],
    "sentence_first": [". ", "\n\n", "\n", " ", ""],
    "line_first": ["\n", "\n\n", ". ", " ", ""],
}

def fixed_chunks(text, chunk_size, chunk_overlap):
    # This is the Step 2 baseline, recreated here so the Step 3 comparison is self-contained.
    splitter = CharacterTextSplitter(
        separator="\n\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=token_len,
        is_separator_regex=False,
    )
    return splitter.split_text(text)

def recursive_chunks(text, chunk_size, chunk_overlap, separators):
    # Recursive splitting tries higher-level separators first, then falls back to smaller ones.
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=token_len,
        separators=separators,
    )
    return splitter.split_text(text)

def show_comparison(label, text):
    print(f"\n=== {label} ===")
    for chunk_size in chunk_sizes:
        base_chunks = fixed_chunks(text, chunk_size, chunk_overlap)
        print(f"fixed_size={chunk_size:4d} | chunks={len(base_chunks):3d} | first={repr(base_chunks[0][:90]) if base_chunks else '[]'}")

        for sep_name, separators in separator_sets.items():
            chunks = recursive_chunks(text, chunk_size, chunk_overlap, separators)
            lengths = [token_len(chunk) for chunk in chunks]
            print(
                f"  recursive/{sep_name:14s} | chunks={len(chunks):3d} | "
                f"avg_tokens={sum(lengths) / len(lengths):7.1f} | "
                f"first={repr(chunks[0][:90]) if chunks else '[]'}"
            )

            # A short sample helps reveal whether sentence or section boundaries were preserved.
            if chunk_size == 500 and sep_name == "paragraph_first" and chunks:
                print(f"    sample_end={repr(chunks[0][-90:])}")

show_comparison("Podcast transcript", podcast_text)
show_comparison("PDF document", pdf_text)

# Step 4: Token-Based Chunking

This section uses `TokenTextSplitter` so chunk sizes align with LLM context windows instead of raw character counts.

In [ ]:
# Compare token-based chunks with character-based chunks using the same nominal sizes.
# The token splitter is the better measure for LLM context windows because it matches how models count input.
from pathlib import Path

import tiktoken
from langchain_text_splitters import CharacterTextSplitter, TokenTextSplitter
from pypdf import PdfReader

transcript_path = Path("data/CVX_manufacturing_podcast_transcript.txt")
pdf_path = Path("data/The AI Ladder.pdf")

assert transcript_path.exists(), f"Missing transcript file: {transcript_path}"
assert pdf_path.exists(), f"Missing PDF file: {pdf_path}"

podcast_text = transcript_path.read_text(encoding="utf-8")
pdf_text = "\n".join(page.extract_text() or "" for page in PdfReader(str(pdf_path)).pages)

# Use the same tokenizer for both the token splitter and the token-count check.
encoding = tiktoken.get_encoding("cl100k_base")
token_len = lambda text: len(encoding.encode(text))

token_sizes = [500, 1000]
overlap = 50

def token_chunks(text, chunk_size, chunk_overlap):
    # This splitter keeps chunk boundaries aligned with token counts.
    splitter = TokenTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        encoding_name="cl100k_base",
    )
    return splitter.split_text(text)

def character_chunks(text, chunk_size, chunk_overlap):
    # This is the character-based comparison point from earlier steps.
    splitter = CharacterTextSplitter(
        separator="\n\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )
    return splitter.split_text(text)

def compare(label, text):
    print(f"\n=== {label} ===")
    for size in token_sizes:
        tok_chunks = token_chunks(text, size, overlap)
        char_chunks = character_chunks(text, size, overlap)

        tok_lengths = [token_len(chunk) for chunk in tok_chunks]
        char_tok_lengths = [token_len(chunk) for chunk in char_chunks]

        print(
            f"token_text_splitter size={size:4d} | chunks={len(tok_chunks):3d} | "
            f"avg_tokens={sum(tok_lengths) / len(tok_lengths):7.1f} | "
            f"first_tokens={tok_lengths[0] if tok_lengths else 0:4d} | "
            f"first={repr(tok_chunks[0][:90]) if tok_chunks else '[]'}"
        )
        print(
            f"character_splitter size={size:4d} | chunks={len(char_chunks):3d} | "
            f"avg_tokens={sum(char_tok_lengths) / len(char_tok_lengths):7.1f} | "
            f"first_tokens={char_tok_lengths[0] if char_tok_lengths else 0:4d} | "
            f"first={repr(char_chunks[0][:90]) if char_chunks else '[]'}"
        )

        # This sample is useful for spotting whether token-based chunking keeps cleaner boundaries.
        if size == 500 and tok_chunks:
            print(f"  token_sample_end={repr(tok_chunks[0][-90:])}")
            print(f"  char_sample_end={repr(char_chunks[0][-90:]) if char_chunks else '[]'}")

compare("Podcast transcript", podcast_text)
compare("PDF document", pdf_text)

# Step 6: Visualize and Compare Results

This section summarizes chunk statistics, plots chunk size distributions, and prints chunk endings so boundary behavior is visible.

In [ ]:
# Build a compact comparison across the three strategies the lab asked for.
# We use Document objects so the histogram code can read chunk.page_content exactly like the example prompt.
from pathlib import Path
from statistics import mean

import matplotlib.pyplot as plt
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter
from pypdf import PdfReader

transcript_path = Path("data/CVX_manufacturing_podcast_transcript.txt")
pdf_path = Path("data/The AI Ladder.pdf")

assert transcript_path.exists(), f"Missing transcript file: {transcript_path}"
assert pdf_path.exists(), f"Missing PDF file: {pdf_path}"

podcast_text = transcript_path.read_text(encoding="utf-8")
pdf_text = "\n".join(page.extract_text() or "" for page in PdfReader(str(pdf_path)).pages)

def make_strategy_chunks(text, strategy_name):
    # Each strategy is intentionally fixed to one representative configuration for easy side-by-side comparison.
    if strategy_name == "Fixed-500":
        splitter = CharacterTextSplitter(
            separator="\n\n",
            chunk_size=500,
            chunk_overlap=0,
            length_function=len,
            is_separator_regex=False,
        )
    elif strategy_name == "Recursive-1000":
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=100,
            length_function=len,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
    elif strategy_name == "Token-500":
        splitter = TokenTextSplitter(
            chunk_size=500,
            chunk_overlap=50,
            encoding_name="cl100k_base",
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy_name}")

    # create_documents returns Document objects, which matches the histogram example in the instructions.
    return splitter.create_documents([text])

def summarize_chunks(chunks):
    sizes = [len(chunk.page_content) for chunk in chunks]
    return {
        "avg": mean(sizes),
        "min": min(sizes),
        "max": max(sizes),
        "count": len(sizes),
        "sizes": sizes,
    }

def print_table(source_name, results):
    print(f"\n{source_name}")
    print(f"{'Strategy':<18} {'Avg chunk size (chars)':>22} {'Min':>8} {'Max':>8} {'# of chunks':>12}")
    for strategy_name, stats in results.items():
        print(
            f"{strategy_name:<18} {stats['avg']:>22.1f} {stats['min']:>8} "
            f"{stats['max']:>8} {stats['count']:>12}"
        )

def show_boundaries(source_name, chunks):
    print(f"\n{source_name} chunk endings")
    # The last 50 characters reveal whether a chunk ends cleanly or cuts off mid-sentence.
    for strategy_name, docs in chunks.items():
        print(f"{strategy_name}:")
        for i, doc in enumerate(docs[:3], start=1):
            print(f"  {i}. {repr(doc.page_content[-50:])}")

def plot_distribution(source_name, chunks):
    # One figure per source keeps the plots readable and makes the three strategies easy to compare.
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, (strategy_name, docs) in zip(axes, chunks.items()):
        sizes = [len(chunk.page_content) for chunk in docs]
        ax.hist(sizes, bins=20)
        ax.set_xlabel("Chunk size (chars)")
        ax.set_ylabel("Frequency")
        ax.set_title(strategy_name)
    fig.suptitle(f"Chunk Size Distribution — {source_name}")
    plt.tight_layout()
    plt.show()

def run_source(source_name, text):
    strategy_order = ["Fixed-500", "Recursive-1000", "Token-500"]
    chunks = {name: make_strategy_chunks(text, name) for name in strategy_order}
    results = {name: summarize_chunks(docs) for name, docs in chunks.items()}

    print_table(source_name, results)
    show_boundaries(source_name, chunks)
    plot_distribution(source_name, chunks)

run_source("Podcast Transcript", podcast_text)
run_source("PDF Document", pdf_text)

## Trade-offs

- `Fixed-500` is simple and predictable, but it is most likely to cut sentences and paragraphs in awkward places.
- `Recursive-1000` is usually the best balance when preserving structure matters, because it prefers larger boundaries first and only falls back when needed.
- `Token-500` is the most aligned with model context windows, but the character lengths vary more, so the chunks can look less uniform on screen.
- The PDF usually benefits more from structure-aware strategies, while the podcast transcript is more sensitive to sentence breaks and uneven conversational turns.

# Step 7: Analyze Chunk Quality

This section checks whether chunks end cleanly at sentence boundaries and compares that signal across both document types.

In [4]:
# Measure how often chunks end with a sentence-ending character.
# This is a simple proxy for boundary preservation: higher percentages usually mean cleaner splits.
from pathlib import Path

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter
from pypdf import PdfReader

transcript_path = Path("data/CVX_manufacturing_podcast_transcript.txt")
pdf_path = Path("data/The AI Ladder.pdf")

assert transcript_path.exists(), f"Missing transcript file: {transcript_path}"
assert pdf_path.exists(), f"Missing PDF file: {pdf_path}"

podcast_text = transcript_path.read_text(encoding="utf-8")
pdf_text = "\n".join(page.extract_text() or "" for page in PdfReader(str(pdf_path)).pages)

def make_strategy_chunks(text, strategy_name):
    # Keep these settings aligned with Step 6 so the quality metric matches the visual comparison.
    if strategy_name == "Fixed-500":
        splitter = CharacterTextSplitter(
            separator="\n\n",
            chunk_size=500,
            chunk_overlap=0,
            length_function=len,
            is_separator_regex=False,
        )
    elif strategy_name == "Recursive-1000":
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=100,
            length_function=len,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
    elif strategy_name == "Token-500":
        splitter = TokenTextSplitter(
            chunk_size=500,
            chunk_overlap=50,
            encoding_name="cl100k_base",
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy_name}")

    return splitter.create_documents([text])

def pct_clean_boundaries(chunks):
    # Empty chunks are ignored so the metric stays well-defined.
    non_empty = [chunk for chunk in chunks if chunk.page_content.strip()]
    if not non_empty:
        return 0.0
    clean = sum(1 for chunk in non_empty if chunk.page_content.strip()[-1] in ".?!")
    return clean / len(non_empty) * 100

def print_quality_table(source_name, text):
    strategies = ["Fixed-500", "Recursive-1000", "Token-500"]
    print(f"\n{source_name}")
    print(f"{'Strategy':<18} {'% clean endings':>16} {'# chunks':>10}")
    for strategy_name in strategies:
        chunks = make_strategy_chunks(text, strategy_name)
        pct = pct_clean_boundaries(chunks)
        print(f"{strategy_name:<18} {pct:>16.1f} {len(chunks):>10}")

        # Print a few endings so you can inspect the exact cut point.
        for i, chunk in enumerate(chunks[:3], start=1):
            print(f"  {i}. {repr(chunk.page_content[-50:])}")

print_quality_table("Podcast Transcript", podcast_text)
print_quality_table("PDF Document", pdf_text)

Created a chunk of size 3464, which is longer than the specified 500



Podcast Transcript
Strategy            % clean endings   # chunks
Fixed-500                     100.0          1
  1. 'is purely informational and not investment advice.'
Recursive-1000                 25.0          4
  1. 'surfactant technology, has been labeled as bullish'
  2. ' they recently reported record-high backlog orders'
  3. 'ion in the next five years, that remains uncertain'
Token-500                      50.0          2
  1. 'rs focused on enhancing its market position within'
  2. 'is purely informational and not investment advice.'

PDF Document
Strategy            % clean endings   # chunks
Fixed-500                      50.0          2
  1. '                                            17\niii'
  2. 'e writes exten‐\nsively on his blog robdthomas.com.'
Recursive-1000                 13.7         51
  1. 'er: Karen Montgomery\nIllustrator: Rebecca Demarest'
  2. ' samples or other technology this work contains or'
  3. '                                                

Recursive chunking usually preserves context best because it tries larger semantic boundaries first, so it is less likely to cut through a sentence or paragraph when a clean break is available. The podcast transcript benefits most from this behavior because conversational text has uneven sentence lengths and more mid-thought transitions, while the PDF benefits because its extracted structure gives the splitter more reliable paragraph and section boundaries.

# Step 8: Make Recommendations

This section summarizes the results and turns them into content-specific recommendations.

In [ ]:
# Build a compact summary table that compares the three strategies on both source types.
# This keeps the recommendation step grounded in the same chunk definitions used above.
from pathlib import Path

from statistics import mean

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter
from pypdf import PdfReader

transcript_path = Path("data/CVX_manufacturing_podcast_transcript.txt")
pdf_path = Path("data/The AI Ladder.pdf")

podcast_text = transcript_path.read_text(encoding="utf-8")
pdf_text = "\n".join(page.extract_text() or "" for page in PdfReader(str(pdf_path)).pages)

def make_strategy_chunks(text, strategy_name):
    if strategy_name == "Fixed-500":
        splitter = CharacterTextSplitter(
            separator="\n\n",
            chunk_size=500,
            chunk_overlap=0,
            length_function=len,
            is_separator_regex=False,
        )
    elif strategy_name == "Recursive-1000":
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=100,
            length_function=len,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
    elif strategy_name == "Token-500":
        splitter = TokenTextSplitter(
            chunk_size=500,
            chunk_overlap=50,
            encoding_name="cl100k_base",
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy_name}")
    return splitter.create_documents([text])

def pct_clean_boundaries(chunks):
    non_empty = [chunk for chunk in chunks if chunk.page_content.strip()]
    if not non_empty:
        return 0.0
    clean = sum(1 for chunk in non_empty if chunk.page_content.strip()[-1] in ".?!")
    return clean / len(non_empty) * 100

def summarize(text):
    rows = []
    for strategy_name in ["Fixed-500", "Recursive-1000", "Token-500"]:
        chunks = make_strategy_chunks(text, strategy_name)
        sizes = [len(chunk.page_content) for chunk in chunks]
        rows.append({
            "strategy": strategy_name,
            "avg": mean(sizes),
            "min": min(sizes),
            "max": max(sizes),
            "count": len(sizes),
            "clean": pct_clean_boundaries(chunks),
        })
    return rows

def print_summary_table(source_name, rows):
    print(f"\n{source_name}")
    print(f"{'Strategy':<18} {'Avg chars':>10} {'Min':>8} {'Max':>8} {'# chunks':>10} {'% clean':>10}")
    for row in rows:
        print(
            f"{row['strategy']:<18} {row['avg']:>10.1f} {row['min']:>8} {row['max']:>8} "
            f"{row['count']:>10} {row['clean']:>10.1f}"
        )

print_summary_table("Podcast Transcript", summarize(podcast_text))
print_summary_table("PDF Document", summarize(pdf_text))

## Recommendations

| Content type | Recommended strategy | Why |
|---|---|---|
| Podcast transcript | `Recursive-1000` | It preserves conversational boundaries better than fixed-size chunking and is less tied to raw token count than `Token-500`.
| PDF document | `Recursive-1000` or `Token-500` | Recursive chunking respects document structure, while token chunking is best when prompt budget matters most.

## Trade-offs

- Use `Fixed-500` only when you need the simplest possible baseline or a fast heuristic.
- Use `Recursive-1000` when preserving local context matters and the source has paragraph or section structure.
- Use `Token-500` when the target model context window is the main constraint and you want the chunk sizes to map cleanly to model limits.
- The podcast transcript is the most fragile source, so it benefits most from recursive splitting. The PDF is structurally richer, so it can tolerate token-based chunking if you need tighter budget control.